# Run `geostt-correct` on Colab GPU with `gemma4:e2b`

This notebook benchmarks the STT post-correction pipeline on a Colab GPU runtime using Ollama and the `gemma4:e2b` model.

**Before you run anything:**
1. `Runtime` → `Change runtime type` → select **GPU** (T4 is fine, L4/A100 even better).
2. Run the cells top-to-bottom.

Typical timings on a free T4:
- `gemma4:e2b` cold start: often ~30–90 s the first time it loads into VRAM.
- Then roughly ~1–5 s per 500-char row (still much faster than CPU).

## 1. Confirm a GPU is attached

In [ ]:
!nvidia-smi

If `nvidia-smi` says *command not found* or `No devices were found`, go back and switch the runtime to GPU before continuing.

## 2. Install `zstd`, then Ollama

Recent Ollama Linux installers unpack a **zstd**-compressed archive. Colab’s base image often does not include `zstd`, which triggers: *This version requires zstd for extraction*. The cell below installs it first, then runs the official installer.

In [ ]:
!apt-get update -qq && apt-get install -y -qq zstd
!curl -fsSL https://ollama.com/install.sh | sh

## 3. Start `ollama serve` in the background
Colab cells block, so we launch the server with `subprocess.Popen` and give it a few seconds to come up.

In [ ]:
import os, subprocess, time, urllib.request

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"

log_path = "/content/ollama.log"
log_file = open(log_path, "w")
proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
    env={**os.environ},
)

for i in range(30):
    try:
        urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=2).read()
        print(f"Ollama is up (after {i+1}s)")
        break
    except Exception:
        time.sleep(1)
else:
    print("Ollama did not start in time. Last log lines:")
    print(open(log_path).read()[-2000:])

## 4. Pull `gemma4:e2b`
First run downloads roughly ~7 GB (quantized); subsequent runs in the same session are instant.

In [ ]:
!ollama pull gemma4:e2b
!ollama list

## 5. Clone the project repo

In [ ]:
%cd /content
![ -d georgian-STT-correction ] || git clone https://github.com/takoshonia/georgian-STT-correction.git
%cd /content/georgian-STT-correction
!git pull --ff-only
!ls -la

## 6. Install Python deps

In [ ]:
%cd /content/georgian-STT-correction
!pip install --quiet openpyxl
!pip install --quiet -e .

## 7. Warm-up: one short generation to load the model into VRAM
The first `/api/generate` call after a fresh `ollama serve` always pays a ~30 s model-load cost. We do it once so the timing below reflects steady-state inference.

In [ ]:
import json, time, urllib.request

payload = json.dumps({
    "model": "gemma4:e2b",
    "prompt": "გამარჯობა",
    "stream": False,
    "options": {"temperature": 0.0, "num_predict": 8},
}).encode("utf-8")

t0 = time.perf_counter()
resp = urllib.request.urlopen(
    urllib.request.Request(
        "http://127.0.0.1:11434/api/generate",
        data=payload,
        headers={"Content-Type": "application/json"},
        method="POST",
    ),
    timeout=300,
).read().decode("utf-8")
print(f"warm-up took {time.perf_counter() - t0:.1f}s")
print(json.loads(resp).get("response", "")[:200])

!ollama ps

`ollama ps` should show `gemma4:e2b` with `100% GPU` (or close to it) in the **PROCESSOR** column. If it says `100% CPU`, the GPU runtime is not actually being used — re-check step 1.

## 8. Run a 20-row benchmark

In [ ]:
import os
os.environ["OLLAMA_MODEL"] = "gemma4:e2b"
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"
os.environ["OLLAMA_TIMEOUT"] = "100"
os.environ["GEOSTT_OLLAMA"] = "1"
os.environ["GEOSTT_HEURISTICS"] = "0"
os.environ["GEOSTT_LEXICON_ENABLE"] = "0"

!python scripts/evaluate_manifest.py \
    --input-json sample_pairs.json \
    --max-rows 20 \
    --output-xlsx /content/report_20rows.xlsx \
    --summary-json /content/report_20rows.json

print("\n--- summary ---")
!cat /content/report_20rows.json

## 9. (Optional) Full 200-row run
On a T4 this should take roughly 5–15 minutes total.

In [ ]:
import os
os.environ["OLLAMA_MODEL"] = "gemma4:e2b"
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11434"
os.environ["OLLAMA_TIMEOUT"] = "100"
os.environ["GEOSTT_OLLAMA"] = "1"
os.environ["GEOSTT_HEURISTICS"] = "0"
os.environ["GEOSTT_LEXICON_ENABLE"] = "0"

!python scripts/evaluate_manifest.py \
    --input-json sample_pairs.json \
    --output-xlsx /content/report_200rows.xlsx \
    --summary-json /content/report_200rows.json \
    --progress-every 10

print("\n--- summary ---")
!cat /content/report_200rows.json

## 10. Download the reports back to your machine

In [ ]:
from google.colab import files

for p in ("/content/report_20rows.xlsx", "/content/report_20rows.json",
         "/content/report_200rows.xlsx", "/content/report_200rows.json"):
    try:
        files.download(p)
    except Exception as e:
        print(f"skip {p}: {e}")